# SQL Analysis – IndiaKart
**Analyst: Vaishali Girase**

The project SQL file is written for MySQL (it uses `DATE_FORMAT`, `STR_TO_DATE`, `YEAR`, `MONTH`). Kaggle uses SQLite, which has no such functions, so the same queries are rewritten with `strftime`. The logic is the same.

**How it works:** we load the 8 CSV files into a small SQLite database, then run SQL with the helper `q("...")`.

In [1]:
import pandas as pd, sqlite3, glob, os
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200

DATA_DIR = "/kaggle/input/datasets/vaishaligirase/indiakart"   # change if your path is different

def load(name):
    return pd.read_csv(glob.glob(os.path.join(DATA_DIR, name + "*.csv"))[0])

conn = sqlite3.connect(":memory:")     # a small database that lives only in memory
for t in ["suppliers","products","inventory","customers","orders","order_items","payments","returns"]:
    df = load(t)
    # SQLite has no DD-MM-YYYY date type, so we store dates as YYYY-MM-DD text
    for c in ["order_date","delivered_date","registration_date","return_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
    df.to_sql(t, conn, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql_query(sql, conn)

print(q("SELECT name FROM sqlite_master WHERE type='table'"))

          name
0    suppliers
1     products
2    inventory
3    customers
4       orders
5  order_items
6     payments
7      returns


# Part A – The 12 queries from the project SQL file

## 1. Monthly revenue trend (GMV)
What it shows: how orders and revenue move month by month.

In [2]:
q("""
SELECT strftime('%Y-%m', order_date) AS month,
       COUNT(order_id)        AS total_orders,
       SUM(final_amount)      AS gross_revenue_inr,
       AVG(final_amount)      AS avg_order_value,
       SUM(CASE WHEN status='Cancelled' THEN 1 ELSE 0 END) AS cancelled_orders
FROM orders
GROUP BY 1 ORDER BY 1
""")

,month,total_orders,gross_revenue_inr,avg_order_value,cancelled_orders
0,2023-06,444,"27,384,848.52","61,677.59",55
1,2023-07,2081,"128,920,508.89","61,951.23",232
2,2023-08,2091,"128,122,036.65","61,273.09",254
3,2023-09,2104,"137,462,482.61","65,333.88",254
4,2023-10,2060,"132,470,873.88","64,306.25",240
5,2023-11,2114,"137,263,358.98","64,930.63",235
6,2023-12,2090,"136,923,402.11","65,513.59",254
7,2024-01,2090,"131,450,024.93","62,894.75",258
8,2024-02,1907,"120,945,866.02","63,422.06",229
9,2024-03,2111,"133,670,054.09","63,320.73",249


## 2. Category-wise sales (cancelled orders excluded)
What it shows: which categories earn the most. Electronics should be on top.

In [3]:
q("""
SELECT oi.category,
       COUNT(DISTINCT oi.order_id) AS orders,
       SUM(oi.quantity)            AS units_sold,
       SUM(oi.total_price)         AS revenue_inr,
       ROUND(SUM(oi.total_price)*100.0/(SELECT SUM(total_price) FROM order_items),2) AS revenue_share_pct
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status NOT IN ('Cancelled')
GROUP BY 1 ORDER BY revenue_inr DESC
""")

,category,orders,units_sold,revenue_inr,revenue_share_pct
0,Electronics,8053,18002,"1,454,623,421.23",50.85
1,Sports & Fitness,7849,17355,"436,109,092.40",15.24
2,Home & Kitchen,8020,17771,"215,623,808.08",7.54
3,Fashion,8009,17767,"131,169,683.05",4.58
4,Automotive,7961,17672,"108,689,675.78",3.80
5,Office Supplies,8070,17819,"81,926,912.42",2.86
6,Toys & Baby,7870,17408,"40,589,038.77",1.42
7,Beauty & Health,7855,17355,"29,159,125.64",1.02
8,Grocery,7998,17611,"16,430,565.09",0.57
9,Books,7911,17459,"12,310,784.10",0.43


## 3. Customer segments
What it shows: size and value of each segment. Compare Premium vs Regular lifetime value (1.47x).

In [4]:
q("""
SELECT segment,
       COUNT(customer_id)     AS customers,
       AVG(total_orders)      AS avg_orders,
       AVG(total_spent)       AS avg_lifetime_value_inr,
       SUM(total_spent)       AS segment_revenue_inr
FROM customers
GROUP BY segment ORDER BY avg_lifetime_value_inr DESC
""")

,segment,customers,avg_orders,avg_lifetime_value_inr,segment_revenue_inr
0,Premium,1501,8.61,"532,735.00","799,635,228.72"
1,Regular,4026,5.73,"361,870.72","1,456,891,522.68"
2,New,1488,4.34,"274,866.48","409,001,320.68"
3,Budget,2505,2.90,"186,178.81","466,377,913.45"
4,Inactive,480,0.56,"32,249.08","15,479,559.05"


## 4. Payment methods and failure rate
What it shows: how customers pay and where payments fail.

In [5]:
q("""
SELECT payment_method,
       COUNT(*) AS transactions,
       SUM(CASE WHEN status='Failed' THEN 1 ELSE 0 END) AS failed_count,
       ROUND(100.0*SUM(CASE WHEN status='Failed' THEN 1 ELSE 0 END)/COUNT(*),2) AS failure_rate_pct
FROM payments
GROUP BY payment_method ORDER BY transactions DESC
""")

,payment_method,transactions,failed_count,failure_rate_pct
0,UPI,17570,624,3.55
1,Credit Card,10007,363,3.63
2,Debit Card,7624,230,3.02
3,Cash on Delivery,5883,209,3.55
4,Net Banking,4026,159,3.95
5,EMI,2943,87,2.96
6,Wallet,1947,76,3.90


## 5. Return rate by category
What it shows: returns linked to orders that contain each category. Note: an order with items from 2 categories is counted in both.

In [6]:
q("""
SELECT oi.category,
       COUNT(DISTINCT r.return_id)  AS returns,
       COUNT(DISTINCT oi.order_id)  AS orders,
       ROUND(100.0*COUNT(DISTINCT r.return_id)/COUNT(DISTINCT oi.order_id),2) AS return_rate_pct
FROM order_items oi
LEFT JOIN returns r ON oi.order_id = r.order_id
GROUP BY 1 ORDER BY return_rate_pct DESC
""")

,category,returns,orders,return_rate_pct
0,Toys & Baby,1934,8993,21.51
1,Sports & Fitness,1919,8984,21.36
2,Home & Kitchen,1930,9123,21.16
3,Books,1862,8977,20.74
4,Automotive,1871,9039,20.70
5,Electronics,1874,9132,20.52
6,Office Supplies,1875,9182,20.42
7,Fashion,1845,9068,20.35
8,Grocery,1841,9116,20.20
9,Beauty & Health,1750,8894,19.68


## 6. Top 10 products by revenue (delivered orders)
What it shows: best selling products.

In [7]:
q("""
SELECT p.product_id, p.product_name, p.category, p.brand,
       SUM(oi.quantity) AS units_sold, SUM(oi.total_price) AS revenue_inr
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o   ON oi.order_id = o.order_id
WHERE o.status = 'Delivered'
GROUP BY 1,2,3,4
ORDER BY revenue_inr DESC LIMIT 10
""")

,product_id,product_name,category,brand,units_sold,revenue_inr
0,PRD0086,Godrej Camera Elite,Electronics,Godrej,181,"27,804,029.78"
1,PRD0067,Ambrane Power Bank Classic,Electronics,Ambrane,160,"25,397,714.76"
2,PRD0062,Godrej Power Bank Standard,Electronics,Godrej,179,"24,406,976.58"
3,PRD0046,Pidilite Headphones Premium,Electronics,Pidilite,142,"23,033,961.12"
4,PRD0055,Reliance Bluetooth Speaker Standard,Electronics,Reliance,150,"22,535,183.43"
5,PRD0054,Zebronics Bluetooth Speaker Standard,Electronics,Zebronics,172,"21,759,074.19"
6,PRD0039,Hero Smart TV Max,Electronics,Hero,150,"21,514,140.94"
7,PRD0018,Infosys Laptop Ultra,Electronics,Infosys,139,"20,923,213.21"
8,PRD0099,Bajaj Router Classic,Electronics,Bajaj,137,"20,334,116.78"
9,PRD0077,Mivi Smartwatch Plus,Electronics,Mivi,136,"19,954,509.26"


## 7. State-wise revenue (delivered orders)
What it shows: which states bring the most revenue.

In [8]:
q("""
SELECT state,
       COUNT(DISTINCT customer_id) AS customers,
       COUNT(order_id)             AS orders,
       SUM(final_amount)           AS revenue_inr,
       AVG(final_amount)           AS aov_inr
FROM orders
WHERE status='Delivered'
GROUP BY state ORDER BY revenue_inr DESC
""")

,state,customers,orders,revenue_inr,aov_inr
0,Uttar Pradesh,1690,6218,"389,642,884.21","62,663.70"
1,Maharashtra,1465,5332,"333,995,100.31","62,639.74"
2,Gujarat,719,2500,"170,976,884.65","68,390.75"
3,Madhya Pradesh,702,2437,"158,579,127.74","65,071.45"
4,Tamil Nadu,676,2359,"150,479,695.71","63,789.61"
5,Karnataka,556,2035,"126,018,629.30","61,925.62"
6,Rajasthan,564,1958,"117,698,055.62","60,111.37"
7,West Bengal,385,1371,"86,935,922.24","63,410.59"
8,Jharkhand,364,1362,"84,229,364.51","61,842.41"
9,Punjab,380,1334,"81,827,377.34","61,339.86"


## 8. Inventory health by warehouse
What it shows: stock status per warehouse (matches your Page 5 dashboard).

In [9]:
q("""
SELECT warehouse_location, status,
       COUNT(*) AS product_count,
       SUM(quantity_available) AS total_units,
       COUNT(CASE WHEN quantity_available < reorder_level THEN 1 END) AS below_reorder
FROM inventory
GROUP BY 1,2 ORDER BY 1,2
""")

,warehouse_location,status,product_count,total_units,below_reorder
0,Bengaluru-WH3,In Stock,154,38490,0
1,Bengaluru-WH3,Low Stock,16,281,16
2,Bengaluru-WH3,Out of Stock,1,0,1
3,Chennai-WH5,In Stock,178,47229,0
4,Chennai-WH5,Low Stock,11,171,11
5,Chennai-WH5,Out of Stock,1,0,1
6,Delhi-WH2,In Stock,154,40103,0
7,Delhi-WH2,Low Stock,11,148,11
8,Hyderabad-WH4,In Stock,149,38870,0
9,Hyderabad-WH4,Low Stock,18,380,18


## 9. Supplier performance
What it shows: how many products each supplier provides and their ratings.

In [10]:
q("""
SELECT s.supplier_id, s.supplier_name, s.category,
       COUNT(p.product_id) AS products_supplied,
       AVG(p.rating)       AS avg_product_rating,
       s.rating            AS supplier_rating
FROM suppliers s
JOIN products p ON s.supplier_id = p.supplier_id
GROUP BY 1,2,3,6
ORDER BY avg_product_rating DESC
LIMIT 15
""")

,supplier_id,supplier_name,category,products_supplied,avg_product_rating,supplier_rating
0,SUP0182,Mishra Offi Pvt Ltd,Office Supplies,2,4.80,3.30
1,SUP0181,Chaudhary Toys Pvt Ltd,Toys & Baby,3,4.70,4.40
2,SUP0053,Sen Fash Pvt Ltd,Fashion,2,4.65,3.90
3,SUP0193,Bajaj Fash Pvt Ltd,Fashion,1,4.60,3.90
4,SUP0127,Dwivedi Offi Pvt Ltd,Office Supplies,2,4.55,3.20
5,SUP0086,Desai Fash Pvt Ltd,Fashion,2,4.55,4.70
6,SUP0159,Verma Offi Pvt Ltd,Office Supplies,3,4.53,4.70
7,SUP0104,Tripathi Fash Pvt Ltd,Fashion,2,4.50,3.90
8,SUP0128,Kapoor Auto Pvt Ltd,Automotive,2,4.50,4.00
9,SUP0168,Bajaj Offi Pvt Ltd,Office Supplies,4,4.50,4.70


## 10. New customers by registration month
What it shows: customer acquisition over time.

In [11]:
q("""
SELECT strftime('%Y-%m', registration_date) AS cohort_month,
       COUNT(customer_id) AS new_customers
FROM customers
WHERE registration_date >= '2023-06-24'
GROUP BY 1 ORDER BY 1
""")

,cohort_month,new_customers
0,2023-06,44
1,2023-07,175
2,2023-08,140
3,2023-09,172
4,2023-10,175
5,2023-11,180
6,2023-12,181
7,2024-01,176
8,2024-02,139
9,2024-03,182


## 11. Channel performance
What it shows: App vs Web vs others, revenue and cancel rate.

In [12]:
q("""
SELECT channel,
       COUNT(order_id)   AS orders,
       SUM(final_amount) AS revenue_inr,
       AVG(final_amount) AS avg_order_value,
       ROUND(SUM(CASE WHEN status='Cancelled' THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS cancel_rate_pct
FROM orders
GROUP BY channel ORDER BY revenue_inr DESC
""")

,channel,orders,revenue_inr,avg_order_value,cancel_rate_pct
0,App,27549,"1,726,659,549.21","62,675.94",12.11
1,Website,14820,"932,976,230.73","62,953.86",11.93
2,Mobile Web,7631,"487,749,764.64","63,916.89",11.68


## 12. Festive season (Oct-Nov) sales
What it shows: whether Diwali months are really higher. Compare with other months.

In [13]:
q("""
SELECT CAST(strftime('%Y', order_date) AS INTEGER) AS year,
       CAST(strftime('%m', order_date) AS INTEGER) AS month,
       COUNT(order_id)   AS orders,
       SUM(final_amount) AS revenue_inr
FROM orders
WHERE CAST(strftime('%m', order_date) AS INTEGER) IN (10,11)
GROUP BY 1,2 ORDER BY 1,2
""")

,year,month,orders,revenue_inr
0,2023,10,2060,"132,470,873.88"
1,2023,11,2114,"137,263,358.98"
2,2024,10,2131,"133,518,340.58"
3,2024,11,2084,"126,264,141.81"


# Part B – My own extra queries
These go beyond the project file. They turn findings from Phase 1 and the dashboard into proof with SQL.

## B1. Failed payments: what happened to those orders?
This is your key finding: all failed payments end in cancelled orders.

In [14]:
q("""
SELECT o.status AS order_status, COUNT(*) AS failed_payments
FROM payments p JOIN orders o ON p.order_id = o.order_id
WHERE p.status = 'Failed'
GROUP BY o.status
""")

,order_status,failed_payments
0,Cancelled,1748


## B2. Value of orders lost to failed payments
Convert the finding into rupees.

In [15]:
q("""
SELECT COUNT(*) AS failed_orders,
       ROUND(SUM(o.final_amount)/10000000.0,2) AS lost_order_value_crore
FROM payments p JOIN orders o ON p.order_id = o.order_id
WHERE p.status = 'Failed'
""")

,failed_orders,lost_order_value_crore
0,1748,10.74


## B3. Payment failure rate by gateway
Which gateway should be reviewed first?

In [16]:
q("""
SELECT gateway, COUNT(*) AS payments,
       ROUND(100.0*SUM(CASE WHEN status='Failed' THEN 1 ELSE 0 END)/COUNT(*),2) AS failure_rate_pct
FROM payments
GROUP BY gateway ORDER BY failure_rate_pct DESC
""")

,gateway,payments,failure_rate_pct
0,Razorpay,7126,3.80
1,CCAvenue,7118,3.72
2,Google Pay,7182,3.69
3,PayU,7121,3.47
4,PhonePe,7170,3.31
5,Paytm,7061,3.31
6,BillDesk,7222,3.17


## B4. Data-quality check: returns on orders that were not delivered
Return requests should not exist for Cancelled or Shipped orders.

In [17]:
q("""
SELECT o.status AS order_status, COUNT(*) AS return_records
FROM returns r JOIN orders o ON r.order_id = o.order_id
GROUP BY o.status ORDER BY return_records DESC
""")

,order_status,return_records
0,Delivered,4327
1,Returned,3933
2,Cancelled,1067
3,Shipped,673


## B5. Data-quality check: orders placed before the customer registered
A logic error in the data.

In [18]:
q("""
SELECT COUNT(*) AS orders_before_registration
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_date < c.registration_date
""")

,orders_before_registration
0,10255


## B6. What-if: cancellation rate from 12% to 10%
Average value of a cancelled order. Estimated saving = (cancelled orders - 10% of all orders) x average cancelled order value. This is an estimate that assumes saved orders look like cancelled ones.

In [19]:
x = q("""
SELECT COUNT(*) AS cancelled_orders, AVG(final_amount) AS avg_cancelled_order_value
FROM orders WHERE status = 'Cancelled'
""")
total_orders = q("SELECT COUNT(*) AS n FROM orders")["n"][0]
extra = x["cancelled_orders"][0] - 0.10*total_orders
print("Orders that would be saved:", round(extra))
print("Estimated revenue saved (Rs Crore): %.1f" % (extra * x["avg_cancelled_order_value"][0] / 1e7))
x

Orders that would be saved: 996
Estimated revenue saved (Rs Crore): 6.1


,cancelled_orders,avg_cancelled_order_value
0,5996,"61,417.84"
